In [ ]:
import numpy as np
import PIL
from IPython.display import Image, display
from torchvision.utils import make_grid, save_image

from dataset import FireSeriesDataset


def norm_01(img):
    return (img - img.min()) / (img.max() - img.min())


ds = FireSeriesDataset("data/images/train")

images, label = ds[0]


def show_image_series(images, filename="temp_series.png"):
    # images: torch tensor of shape (N, C, H, W)
    grid = make_grid(images, nrow=images.shape[0], normalize=True)
    save_image(grid, filename)
    display(Image(filename))


show_image_series(images)

print(ds.label2name[label])


In [ ]:
import matplotlib.pyplot as plt


def display_images_from_array(images_array, max_cols=5, figsize=(15, 10)):
    """
    Displays a NumPy array of multiple images in a Jupyter Notebook.

    The images are assumed to be stacked along the first dimension.

    Args:
        images_array (np.ndarray): A NumPy array where the first dimension 
                                   is the number of images (N, H, W, C) or (N, H, W).
        max_cols (int): The maximum number of columns to use for displaying the images.
        figsize (tuple): A tuple (width, height) specifying the figure size in inches.
    """
    
    # Check if the array is empty
    if images_array is None or images_array.size == 0:
        print("The input array is empty.")
        return

    # Determine the number of images
    num_images = images_array.shape[0]

    # Calculate the number of rows and columns for the subplot grid
    # Use min(num_images, max_cols) to avoid a single column for few images
    cols = min(num_images, max_cols) 
    rows = int(np.ceil(num_images / cols))
    
    # Create the figure and a set of subplots
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    
    # Flatten the axes array for easy iteration, especially if rows or cols is 1
    if num_images == 1:
        # If there's only one image, axes isn't an array but the subplot object itself
        axes = np.array([axes]) 
    else:
        # Otherwise, flatten the 2D array of axes
        axes = axes.flatten()

    # Iterate through each image and display it
    for i in range(num_images):
        ax = axes[i]
        
        # Display the image
        # 'images_array[i]' is the i-th image (H, W, C) or (H, W)
        ax.imshow(images_array[i], cmap='gray' if images_array[i].ndim == 2 else None) 
        
        # Set the title
        ax.set_title(f"Image {i+1}")
        
        # Turn off axis ticks and labels for a cleaner look
        ax.axis('off')

    # Remove any unused subplots if the number of images doesn't fill the last row
    for j in range(num_images, rows * cols):
        fig.delaxes(axes[j])
        
    # Adjust layout to prevent titles from overlapping
    plt.tight_layout()
    
    # Display the plot
    plt.show()

images = images.permute((0,2,3,1))
images = norm_01(images)
images = images * 255
images = images.numpy().astype(np.uint8) 
display_images_from_array(images)

### Calculate LBP on single frames

In [ ]:
import cv2
import torchvision.transforms.functional as F
from skimage.feature import local_binary_pattern

radius = 3
n_points = 8 * radius

img = images[0]
img = norm_01(img)
F.to_pil_image(img)

In [ ]:
def to_uint8(image):
    return (image.permute(1,2,0).numpy()*255).astype(np.uint8)

def to_numpy(image):
    return (image.permute(1,2,0).numpy())

def to_grayscale(image):
    image = norm_01(image)
    image = to_uint8(image)
    return np.array(PIL.Image.fromarray(image).convert("L"))

def get_lbp(image):
    lbp = local_binary_pattern(image, n_points, radius)
    return norm_01(lbp)

images_gray = []
images_lbp = []
for image in images:
    gs_image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    lbp = get_lbp(gs_image.squeeze())
    images_lbp.append(lbp)
    images_gray.append(gs_image)
display_images_from_array(np.array(images_lbp))

In [ ]:
h, w = images[0].shape[:2]
motion_image = np.zeros((h, w, 3), dtype=np.uint8)
# Optical flow is now calculated
flow = cv2.calcOpticalFlowFarneback(images_gray[0], images_gray[1], None, 0.7, 3, 15, 5, 5, 1.2, 0)
# Compute magnite and angle of 2D vector
mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
# Set image hue value according to the angle of optical flow
motion_image[..., 0] = ang * 180 / np.pi / 2
# Set value as per the normalized magnitude of optical flow
motion_image[..., 1] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

img_gray = images_gray[3].copy()
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
img_gray = clahe.apply(img_gray)

motion_image[...,2] = img_gray.squeeze()
# Convert to rgb
rgb_representation = cv2.cvtColor(motion_image, cv2.COLOR_HSV2RGB)
PIL.Image.fromarray((rgb_representation).astype(np.uint8))

In [ ]:
def get_optical_flow_map(prev, next):
    """prev and next should be np.array with shape (H,W) and range [0,255], dtype=np.uint8
    
    # """
    prev = cv2.GaussianBlur(prev, (5,5), 1)
    next = cv2.GaussianBlur(next, (5,5), 1)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    prev = clahe.apply(prev)
    next = clahe.apply(next)

    # Optical flow is now calculated
    flow = cv2.calcOpticalFlowFarneback(prev, next, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    # Compute magnite and angle of 2D vector
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    # Set image hue value according to the angle of optical flow
    ang_map = ang * 180 / np.pi / 2
    # Set value as per the normalized magnitude of optical flow
    mag_map = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    # Convert to rgb
    # rgb_representation = cv2.cvtColor(motion_image, cv2.COLOR_HSV2RGB)
    return ang_map, mag_map

# images_gray_np = [img.numpy().squeeze() for img in images_gray]

ang, mag = get_optical_flow_map(images_gray[0], images_gray[1])
motion_image = np.concatenate((ang[...,None], mag[...,None], (images_lbp[0][...,None]*255)), axis=2)
motion_image = motion_image.astype(np.uint8)

rgb_representation = cv2.cvtColor(motion_image, cv2.COLOR_HSV2RGB)
PIL.Image.fromarray((rgb_representation).astype(np.uint8))

In [ ]:
ang, mag = get_optical_flow_map(images_lbp[0].squeeze(), images_lbp[1].squeeze())
motion_image = np.concatenate((ang[...,None], mag[...,None], images_lbp[0][...,None]*255), axis=2)
motion_image = motion_image.astype(np.uint8)

rgb_representation = cv2.cvtColor(motion_image, cv2.COLOR_HSV2RGB)
PIL.Image.fromarray((rgb_representation).astype(np.uint8))